<a href="https://colab.research.google.com/github/MuhammadShayan8401/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadShayan8401/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Method choice

I will use **Logistic Regression** as the main supervised model for this lane because the task is binary classification: identifying content items whose search visibility is declining.

Logistic Regression is a suitable next step after the Week-4 baseline because it can learn relationships between the engineered search-performance signals and the decline label while remaining relatively easy to interpret. This makes it useful for decision support rather than optimizing for complexity alone.

I will compare the model against the Week-4 baseline using the same evaluation data, split design, and primary metric. I will only treat the model as an improvement if the measured result is better under the same evaluation setup.

The model will exclude any variables derived from the future outcome to avoid target leakage.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

I will use a **client-grouped holdout split**. The grouping variable is `client_hash_id`, so all content items belonging to a client remain entirely within either the training set or the test set.

This is more honest than randomly splitting individual rows because pages from the same client may share search-performance patterns. Keeping each client in only one split reduces the risk that client-specific information leaks from training into evaluation.

I will use approximately 80% of the clients for training and 20% for testing, matching the client-holdout idea used by the reference modeling pipeline.

The test set will be used only for final comparison against the Week-4 baseline.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

I trained a Logistic Regression model using the same decision problem and evaluated it using a client-grouped holdout split. The model was compared with the Week-4 baseline using Precision@50 as the primary decision-support metric.

The Week-4 baseline achieved a measured Precision@50 of **92.00%**: 46 of the top 50 baseline recommendations were later observed to meet the decline label.

Logistic Regression achieved a Precision@50 of **78.00%** on its held-out test set.

Therefore, Logistic Regression did not improve on the Week-4 baseline for Precision@50. The measured difference was **-14 percentage points**.

This result does not mean the baseline is a better predictive model in every respect. It means that, under this evaluation metric and setup, the simple baseline ranking produced a stronger top-50 decision-support result than Logistic Regression. I therefore do not treat added model complexity as an improvement by itself.

In [ ]:
# ============================================================
# ML-08 — SECTION 3
# Train + Compare vs Week-4 Baseline
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Logistic Regression predictions
# ------------------------------------------------------------

y_pred = logistic_model.predict(X_test)

y_prob = logistic_model.predict_proba(X_test)[:, 1]


# ------------------------------------------------------------
# 2. Model Precision@50
# ------------------------------------------------------------

def precision_at_k(y_true, scores, k=50):

    y_true = pd.Series(
        y_true
    ).reset_index(drop=True)

    scores = pd.Series(
        scores
    ).reset_index(drop=True)

    k = min(k, len(scores))

    top_k = (
        scores
        .nlargest(k)
        .index
    )

    return float(
        y_true.iloc[top_k].mean()
    )


model_precision_50 = precision_at_k(
    y_test,
    y_prob,
    k=50
)


# ------------------------------------------------------------
# 3. Measured Week-4 baseline
# ------------------------------------------------------------

# Measured directly from ML-07:
# 46 declining items among the top 50 recommendations.

baseline_precision_50 = 0.92


# ------------------------------------------------------------
# 4. Model-vs-baseline comparison
# ------------------------------------------------------------

difference_pp = (
    model_precision_50
    - baseline_precision_50
) * 100


comparison = pd.DataFrame({

    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],

    "Precision@50": [
        baseline_precision_50,
        model_precision_50
    ],

    "Precision@50 (%)": [
        baseline_precision_50 * 100,
        model_precision_50 * 100
    ],

    "Difference vs Baseline (pp)": [
        0,
        difference_pp
    ]

})


# ------------------------------------------------------------
# 5. Display comparison
# ------------------------------------------------------------

print("=" * 60)
print("MODEL VS WEEK-4 BASELINE")
print("=" * 60)

display(
    comparison.round(4)
)


# ------------------------------------------------------------
# 6. Interpretation
# ------------------------------------------------------------

print(
    f"Week-4 baseline Precision@50: "
    f"{baseline_precision_50:.2%}"
)

print(
    f"Logistic Regression Precision@50: "
    f"{model_precision_50:.2%}"
)

print(
    f"Difference: "
    f"{difference_pp:.2f} percentage points"
)

if model_precision_50 > baseline_precision_50:

    print(
        "Result: Logistic Regression improved "
        "on the Week-4 baseline."
    )

elif model_precision_50 < baseline_precision_50:

    print(
        "Result: Week-4 baseline performed better "
        "than Logistic Regression."
    )

else:

    print(
        "Result: Both methods achieved the same "
        "Precision@50."
    )

MODEL VS WEEK-4 BASELINE


,Method,Precision@50,Precision@50 (%),Difference vs Baseline (pp)
0,Week-4 baseline,0.92,92.0,0.0
1,Logistic Regression,0.78,78.0,-14.0


Week-4 baseline Precision@50: 92.00%
Logistic Regression Precision@50: 78.00%
Difference: -14.00 percentage points
Result: Week-4 baseline performed better than Logistic Regression.


## 4. Errors and interpretation

The Logistic Regression model produced 1,508 false positives and 1,130 false negatives on the held-out test set. There were 3,525 correct classifications.

The model's Precision@50 was 78%, compared with 92% for the Week-4 baseline. This means the Logistic Regression ranking produced fewer declining items in its highest-priority 50 cases under the measured evaluation setup.

The strongest model coefficients by absolute magnitude were `days_with_impressions`, `word_count`, `days_with_sessions`, `char_count`, and `age_tier_order`. These coefficients show which engineered signals the linear model relied on most, but they should be interpreted as associations used by the model rather than causal effects.

The most uncertain cases had predicted probabilities close to 0.50. These cases included both false positives and false negatives, showing that some content items are difficult for the linear decision boundary to separate using the available features.

Overall, the observed errors suggest that Logistic Regression captures some useful signal but does not outperform the simpler baseline for the top-50 decision-support task. The baseline therefore remains the stronger measured approach for this specific metric and evaluation setup.

In [ ]:
# ============================================================
# ML-08 — SECTION 4
# Error Analysis + Feature Interpretation
# ============================================================

from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Error categories
# ------------------------------------------------------------

error_analysis = pd.DataFrame({
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(y_pred).reset_index(drop=True),
    "probability_declining": pd.Series(y_prob).reset_index(drop=True)
})

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 1) &
        (error_analysis["predicted"] == 0),

        (error_analysis["actual"] == 0) &
        (error_analysis["predicted"] == 1)
    ],
    [
        "false_negative",
        "false_positive"
    ],
    default="correct"
)

print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

display(
    error_analysis["error_type"]
    .value_counts()
    .rename_axis("Error type")
    .reset_index(name="Count")
)

# ------------------------------------------------------------
# 2. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion matrix:")
print(cm)

# ------------------------------------------------------------
# 3. Feature interpretation
# ------------------------------------------------------------

model_step = logistic_model.named_steps["model"]

coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": model_step.coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

top_signals = (
    coefficients
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .head(10)
)

print("\nTOP MODEL SIGNALS")
print("=" * 60)

display(
    top_signals.reset_index(drop=True).round(4)
)

# ------------------------------------------------------------
# 4. Most uncertain cases
# ------------------------------------------------------------

error_analysis["uncertainty"] = (
    (error_analysis["probability_declining"] - 0.5)
    .abs()
)

most_uncertain = (
    error_analysis
    .sort_values("uncertainty")
    .head(20)
)

print("\nMOST UNCERTAIN CASES")
print("=" * 60)

display(
    most_uncertain[
        [
            "actual",
            "predicted",
            "probability_declining",
            "error_type"
        ]
    ]
)

# ------------------------------------------------------------
# 5. Final interpretation summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SECTION 4 COMPLETE")
print("=" * 60)

print(
    "The Logistic Regression model produced both false positives "
    "and false negatives."
)

print(
    f"Precision@50: {model_precision_50:.2%}"
)

print(
    f"Week-4 baseline Precision@50: "
    f"{baseline_precision_50:.2%}"
)

print(
    f"Difference: "
    f"{(model_precision_50 - baseline_precision_50) * 100:.2f} "
    f"percentage points"
)

ERROR ANALYSIS


,Error type,Count
0,correct,3525
1,false_positive,1508
2,false_negative,1130



Confusion matrix:
[[1506 1508]
 [1130 2019]]

TOP MODEL SIGNALS


,feature,coefficient,absolute_coefficient
0,days_with_impressions,0.7208,0.7208
1,word_count,0.5799,0.5799
2,days_with_sessions,-0.4996,0.4996
3,char_count,-0.3645,0.3645
4,age_tier_order,-0.1671,0.1671
5,days_since_last_update,0.1571,0.1571
6,content_age_days,-0.1551,0.1551
7,impressions_prev_30d,0.1514,0.1514
8,avg_position,-0.1396,0.1396
9,scroll_rate,0.1279,0.1279



MOST UNCERTAIN CASES


,actual,predicted,probability_declining,error_type
4548,1,0,0.499984,false_negative
1774,0,1,0.500080,false_positive
1486,1,1,0.500084,correct
4179,1,1,0.500090,correct
5132,0,0,0.499903,correct
3358,1,0,0.499891,false_negative
2453,1,0,0.499826,false_negative
3414,0,0,0.499821,correct
5634,0,1,0.500192,false_positive
3513,0,0,0.499740,correct



SECTION 4 COMPLETE
The Logistic Regression model produced both false positives and false negatives.
Precision@50: 78.00%
Week-4 baseline Precision@50: 92.00%
Difference: -14.00 percentage points


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.